##### Single Ingest

In [1]:
pip install pyroaring

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyroaring import BitMap

In [3]:
# doc1 = {
#     "name": "Alex",
#     "theme": "dark",
#     "country": "USA"
# }

# doc2 = {
#     "name": "John",
#     "theme": "dark",
#     "country": "India"
# }

# doc3 = {
#     "name": "Alex",
#     "theme": "light",
#     "country": "India"
# }


# engine = SearchEngine()

# engine.index_document(doc1)
# engine.index_document(doc2)
# engine.index_document(doc3)

In [4]:
# doc4 = {
#     "name": "Pragya",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc4)

In [5]:
# print(engine.get_cardinality())

In [6]:
# engine.disable_high_cardinality(2)

In [7]:
# doc5 = {
#     "name": "Rachna",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc5)

In [8]:
#name not updated becoz name cardinality false
# print(engine.get_cardinality())

In [9]:
# engine.lookup("name", "Alex")
# {0,2}

In [10]:
# engine.query_and([
#     engine.lookup("name","Alex"),
#     engine.lookup("theme","dark")
# ])
# {0}

In [11]:
# engine.query_or([
#     engine.lookup("name","Alex"),
#     engine.lookup("country","India")
# ])

In [12]:
# engine.query_not(
#     engine.lookup("country","USA")
# )
# {1,2}

In [13]:
# engine.query_in(
#     "country",
#     ["USA","India"]
# )
# {0,1,2}

In [14]:
# engine.query_not_in(
#     "theme",
#     ["dark"]
# )
# {2}

### BSI

In [15]:
docs = [
    {"id": 0, "age": 5},
    {"id": 1, "age": 3},
    {"id": 2, "age": 6},
    {"id": 3, "age": 1},
    {"id": 4, "age": 7},
    {"id": 5, "age": 4},
    {"id": 6, "age": 2},
    {"id": 7, "age": 0}
]

| Doc | Age | Binary |
| --: | --: | :----: |
|   0 |   5 |   101  |
|   1 |   3 |   011  |
|   2 |   6 |   110  |
|   3 |   1 |   001  |
|   4 |   7 |   111  |
|   5 |   4 |   100  |
|   6 |   2 |   010  |
|   7 |   0 |   000  |


Look at the leftmost bit.Which document IDs have bit 2 = 1? 

0 2 4 5

similarly, Slice1 = BitMap([1,2,4,6])

and Slice0 = BitMap([0,1,3,4])

In [16]:
from datetime import datetime,date
import re

In [17]:
class BSI:
  def __init__(self):
    self.slices=[]
    self.all_docs=BitMap()

  def ensure_slices(self,value):
    bits = max(1, value.bit_length())
    while len(self.slices)<bits:
      self.slices.append(BitMap())

  def add(self,doc_id,value):
    self.ensure_slices(value)
    self.all_docs.add(doc_id)
    for i in range(len(self.slices)):
      if ((value>>i)&1)==1:       #if the i'th bit is 1 we add the document to slice[i]
        self.slices[i].add(doc_id)

  def print_slices(self):
    for i, bitmap in enumerate(self.slices):
        print(f"Slice {i}: {list(bitmap)}")

  def equals(self, value):
    if value.bit_length() > len(self.slices):
        return BitMap()
    answer = self.all_docs.copy()
    bits = max(len(self.slices), value.bit_length())
    for i in range(bits):
      if ((value >> i) & 1) == 1:
        answer &= self.slices[i]
      else:
        zero_bitmap = self.all_docs - self.slices[i]#documents which have this bit zero
        answer &= zero_bitmap
    return answer

  def greater_than(self,value):
    if value.bit_length() > len(self.slices):
      for i in range(len(self.slices), value.bit_length()):
        if ((value >> i) & 1):
            return BitMap()
    equal=self.all_docs.copy()#initially all documents present here
    greater=BitMap()#initially empty
    for i in reversed(range(len(self.slices))):
      bit=(value>>i)&1
      if bit==1:
      #eg if ith  bit of value is 1; out of all documents only documents whose ith 1 can be tied and later be > value
      #who has ith bit 1 in this case? slice[i] so we intersect from that
        equal&=self.slices[i]
      else:#if i'th bit is zero
        #out of all documents
        #documents with i'th bit 1 automatically become greater
        #remaining all leftover docs must have ith bit 0
        greater|=equal&self.slices[i]
        zero_bitmap=self.all_docs-self.slices[i]
        equal&=zero_bitmap
    return greater

  def greater_than_equals(self,value):
    return self.greater_than(value)|self.equals(value)

  def less_than(self,value):
    return self.all_docs-(self.greater_than(value)|self.equals(value))

  def less_than_equals(self,value):
    return self.less_than(value)|self.equals(value)

  def get_value(self, doc_id): #returns the original value of a document
    value=0
    for i in range(len(self.slices)):
      if(self.slices[i].contains(doc_id)):
        value|=(1<<i)
    return value


In [18]:
bsi = BSI()

for doc in docs:
    bsi.add(doc["id"], doc["age"])


In [19]:
bsi.print_slices()

Slice 0: [0, 1, 3, 4]
Slice 1: [1, 2, 4, 6]
Slice 2: [0, 2, 4, 5]


In [20]:
print(bsi.greater_than(5))

BitMap([2, 4])


In [21]:
print(bsi.greater_than(2))

BitMap([0, 1, 2, 4, 5])


In [22]:
print(bsi.greater_than(7))

BitMap([])


In [23]:
print(bsi.less_than_equals(5))

BitMap([0, 1, 3, 5, 6, 7])


In [24]:
class QueryNode:
        def __init__(self,operator=None,left=None,right=None,field=None,value=None,comparison=None):
          self.operator = operator
          self.left = left
          self.right = right
          self.field = field #condition=("age", ">", 25)
          self.value = value
          self.comparison = comparison

        def is_leaf(self):
          return self.field is not None


In [25]:
from datetime import datetime,date

class SearchEngine:
    def __init__(self, ignored_fields=None):
        self.schema = {}
        self.index = {}#categorical values
        self.bsi_index={}#numerical values
        self.documents = {}
        self.all_docs=BitMap();
        self.next_doc_id = 0
        self.float_scale=1000
        self.ignored_fields = {
            field.lower()
            for field in (ignored_fields or set())
        }
        self.KEYWORDS={"NOT IN", "BETWEEN", "AND", "OR", "NOT", "IN"}
        self.VALID_OPERATORS = {"=","==","!=",">",">=","<","<=","IN","NOT IN","BETWEEN"}
        self.tokens = []
        self.current = 0
        self.field_present = {}

    # Determine field type
    def determine_type(self, value):
        if isinstance(value, bool):
            return "bool"
        elif isinstance(value, int):
            return "int"
        elif isinstance(value, float):
            return "float"
        elif isinstance(value, str):
            try:
                datetime.strptime(value, "%Y-%m-%d")
                return "date"
            except ValueError:
                return "categorical"
        return "unknown"

    # Build schema
    def build_schema(self, doc):
        for key, value in doc.items():
            if key.lower() in self.ignored_fields:
                continue
            if key in self.schema:
                continue
            field_type = self.determine_type(value)
            if field_type == "unknown":
                raise ValueError(f"Unsupported field type for {key}")
            if (field_type == "int" or field_type=="float") and value < 0:
                raise ValueError("Negative integers are not supported.")
            self.schema[key] = {
                "type": field_type,
                "bitmap": field_type in ("categorical", "bool"),
                "bsi": field_type in ("int", "float", "date"),
            }

    def encode_numeric(self, field, value):
        field_type = self.schema[field]["type"]
        if field_type == "date":
            return datetime.strptime(value, "%Y-%m-%d").toordinal()
        if field_type == "float":
            return int(round(value * self.float_scale))
        return value

    # Index one document
    def index_document(self, doc):
        # Update schema with any newly discovered fields
        self.build_schema(doc)
        doc_id = self.next_doc_id
        #vaildation
        for field, value in doc.items():
            # Ignore fields that aren't indexed
            if field.lower() in self.ignored_fields:
                continue

            field_type = self.determine_type(value)

            if field_type=="unknown":
                raise ValueError(f"Unsupported type for field {field}")

            if field not in self.schema:
                raise ValueError(f"Unknown field {field}")

            if field_type != self.schema[field]["type"]:
                raise ValueError(f"{field} should be {self.schema[field]['type']}")
            if field_type in ("int", "float") and value < 0:
                raise ValueError("Negative integers are not supported.")
            if field not in self.field_present:
                self.field_present[field] = BitMap()
            self.field_present[field].add(doc_id)

        # Save original document
        self.documents[doc_id] = doc
        self.all_docs.add(doc_id)

        #index
        for field, value in doc.items():
            if field.lower() in self.ignored_fields:
                continue
            #BSI
            if self.schema[field]["bsi"]:
                if field not in self.bsi_index:
                    self.bsi_index[field] = BSI()
                encoded_value = self.encode_numeric(field, value)
                self.bsi_index[field].add(doc_id, encoded_value)
            #bitmap
            if not self.schema[field]["bitmap"]:
                continue
            if field not in self.index:
                self.index[field] = {}
            if value not in self.index[field]:
                self.index[field][value] = BitMap()
            self.index[field][value].add(doc_id)
        self.next_doc_id += 1

    def validate_query_value(self, field, value):
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        expected = self.schema[field]["type"]
        actual = self.determine_type(value)
        if actual in ("int", "float") and value < 0:
            raise ValueError("Negative numeric values are not supported.")
        if actual != expected:
            raise ValueError(f"Field '{field}' expects {expected}, got {actual}")
    # Equality lookup
    def lookup(self, field, value):## returns bitmap
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        if field not in self.index:
            return BitMap()
        if value not in self.index[field]:
            return BitMap()
        return self.index[field][value]

    def query_and(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer &= result
        return answer

    def query_or(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer |= result
        return answer

    def query_not(self, result):
        return self.all_docs - result

    def query_in(self, field, values):
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        if self.schema[field]["bsi"]:
            if field not in self.bsi_index:
                return BitMap()
            results = []
            for value in values:
                self.validate_query_value(field, value)
                value = self.encode_numeric(field, value)
                results.append(self.bsi_index[field].equals(value))#BSI returns bitmap of matching documents
            return self.query_or(results)#which is why query or works here
        else:
            results = []
            for value in values:
                results.append(self.lookup(field, value))
            return self.query_or(results)

    def query_not_in(self, field, values):#works for both categorical and numeric
        result = self.query_in(field, values)
        return self.query_not(result)

    # batch ingest
    def index_documents(self, docs):
        for doc in docs:
            self.index_document(doc)

    def greater_than(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].greater_than(value)

    def greater_than_equals(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].greater_than_equals(value)

    def less_than(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].less_than(value)

    def less_than_equals(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].less_than_equals(value)

    def equals_numeric(self,field,value):
        self.validate_query_value(field, value)
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.encode_numeric(field, value)
        return self.bsi_index[field].equals(value)

    def between(self, field, low, high):
        if low > high:
            raise ValueError("low must be <= high")
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        self.validate_query_value(field, low)
        self.validate_query_value(field, high)
        low = self.encode_numeric(field, low)
        high = self.encode_numeric(field, high)
        return (self.bsi_index[field].greater_than_equals(low) & self.bsi_index[field].less_than_equals(high))

    def get_value(self, field, doc_id):
        if field not in self.bsi_index:
            raise ValueError(f"{field} is not a numeric field")
        value = self.bsi_index[field].get_value(doc_id)
        if self.schema[field]["type"] == "date":
            return date.fromordinal(value).strftime("%Y-%m-%d")
        elif self.schema[field]["type"] == "float":
            return value / self.float_scale
        return value

## QUERY PARSING
    def tokenize(self, query):
        pattern = r'"[^"]*"|\'[^\']*\'|\bNOT\s+IN\b|\bBETWEEN\b|\bAND\b|\bOR\b|\bNOT\b|\bIN\b|>=|<=|!=|==|=|>|<|\(|\)|,|[A-Za-z0-9_.-]+'
        # Detect unterminated quotes before tokenizing -- check each quote
        # style independently, since one could be balanced while the other isn't.
        if query.count('"') % 2 != 0:
            raise ValueError(f"Unterminated double-quoted string in query: {query!r}")
        if query.count("'") % 2 != 0:
            raise ValueError(f"Unterminated single-quoted string in query: {query!r}")
        pos = 0
        tokens = []
        for match in re.finditer(pattern, query, flags=re.IGNORECASE):
            # Catch characters between matches that the pattern doesn't cover
            # (e.g. *, %, $) instead of silently skipping them.
            gap = query[pos:match.start()]
            if gap.strip():
                raise ValueError(f"Unrecognized character(s) {gap.strip()!r} in query: {query!r}")
            pos = match.end()
            token = match.group()
            token = token.strip()
            if (token.startswith('"') and token.endswith('"')) or \
               (token.startswith("'") and token.endswith("'")):
                token = token[1:-1]
            else:
                # Collapse internal whitespace (handles "NOT   IN" -> "NOT IN")
                token = re.sub(r'\s+', ' ', token)
                # Normalize keyword casing so downstream comparisons are reliable
                if token.upper() in self.KEYWORDS:
                    token = token.upper()
            tokens.append(token)
        # Check for trailing unrecognized characters after the last match
        trailing = query[pos:]
        if trailing.strip():
            raise ValueError(f"Unrecognized character(s) {trailing.strip()!r} in query: {query!r}")
        return tokens

    def peek(self):
        if self.current >= len(self.tokens):
            return None
        return self.tokens[self.current]
    def consume(self):
        token = self.peek()
        if token is None:
            raise ValueError("Unexpected end of query")
        self.current += 1
        return token

    def parse(self, query):
        self.tokens = self.tokenize(query)
        self.current = 0
        root = self.parse_expression()
        if self.peek() is not None:
            raise ValueError(f"Unexpected token '{self.peek()}'")
        return root

    def parse_expression(self):
        return self.parse_or()

    def parse_or(self):
        node = self.parse_and()#parse everything on the left side fist as and has higher presedence
        while self.peek() == "OR":
            self.consume()
            right = self.parse_and()#parse everything on the right side
            node = QueryNode(operator="OR",left=node,right=right)
        return node

    def parse_and(self):
        node = self.parse_not()#parse everything on the left side fist as and has higher presedence
        while self.peek() == "AND":
            self.consume()
            right = self.parse_not()#parse everything on the right side
            node = QueryNode(operator="AND",left=node,right=right)
        return node

    def parse_not(self):
        if self.peek() == "NOT":
          self.consume()
          return QueryNode(operator="NOT",left=self.parse_not())
        return self.parse_primary()

    def parse_primary(self):
        if self.peek() == "(":
          self.consume()
          node = self.parse_expression()
          if self.peek() != ")":
            raise ValueError("Expected ')'")
          self.consume()
          return node
        return self.parse_condition()

    def parse_condition(self):
        field = self.consume()
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        operator = self.consume()
        if operator not in self.VALID_OPERATORS:
            raise ValueError(f"Unknown operator '{operator}'")
        # BETWEEN
        if operator == "BETWEEN":
          low = self.consume()
          if self.consume() != "AND":
            raise ValueError("Expected AND inside BETWEEN")
          high = self.consume()
          low = self.convert_value(field, low)
          high = self.convert_value(field, high)
          return QueryNode(field=field,comparison="BETWEEN",value=(low, high))
        # IN / NOT IN
        elif operator in ("IN", "NOT IN"):
          if self.consume() != "(":
            raise ValueError("Expected '('")
          values = []
          while True:
            values.append(self.convert_value(field, self.consume()))
            token = self.consume()
            if token == ")":
              break
            if token != ",":
              raise ValueError("Expected ',' or ')'")
          return QueryNode(field=field,comparison=operator,value=values)
        # Normal comparison
        else:
          value = self.consume()
          value = self.convert_value(field, value)
          return QueryNode(field=field,comparison=operator,value=value)

    def convert_value(self, field, value):
        field_type = self.schema[field]["type"]
        if field_type == "int":
          return int(value)
        elif field_type == "float":
          return float(value)
        elif field_type == "bool":
          if value.lower() == "true":
            return True
          if value.lower() == "false":
            return False
          raise ValueError("Boolean must be true or false")
        return value

    def evaluate(self, node):
        if node.is_leaf():
          return self.execute_condition(node.field,node.comparison,node.value)
        if node.operator == "AND":
          return self.query_and([self.evaluate(node.left),self.evaluate(node.right)])
        if node.operator == "OR":
          return self.query_or([self.evaluate(node.left),self.evaluate(node.right)])
        if node.operator == "NOT":
          return self.query_not(self.evaluate(node.left))
        raise ValueError("Unknown node")

    def search(self, query):
        tree = self.parse(query)
        bitmap = self.evaluate(tree)
        return self.fetch(bitmap)


    def execute_condition(self, field, operator, value):
        if field not in self.schema:
            raise ValueError(f"Unknown field '{field}'")
        # Numeric / Date fields
        if self.schema[field]["bsi"]:
            if operator in ("=","=="):
                return self.equals_numeric(field, value)
            elif operator == "!=":
                return self.query_not(self.equals_numeric(field, value))
            elif operator == ">":
                return self.greater_than(field, value)
            elif operator == ">=":
                return self.greater_than_equals(field, value)
            elif operator == "<":
                return self.less_than(field, value)
            elif operator == "<=":
                return self.less_than_equals(field, value)
            elif operator == "IN":
                return self.query_in(field, value)
            elif operator == "NOT IN":
                return self.query_not_in(field, value)
            elif operator == "BETWEEN":
                low, high = value
                return self.between(field, low, high)
            else:
                raise ValueError(f"{operator} is not supported for numerical fields")
        # Categorical fields
        else:
            if operator in ("=","=="):
                return self.lookup(field, value)
            elif operator == "!=":
                self.validate_query_value(field, value)
                return self.query_not(self.lookup(field, value))
            elif operator == "IN":
                return self.query_in(field, value)
            elif operator == "NOT IN":
                return self.query_not_in(field, value)
            else:
                raise ValueError(f"{operator} is not supported for categorical fields")

    def fetch(self, bitmap):#converts bitmap into actual documents as bitmaps return the doc_id
        docs=[]
        for doc_id in bitmap:#bitmap is the result of a query -->works on bsi too as bsi returns bitmap of doc_ids
            docs.append(self.documents[doc_id])
        return docs


    #cardinality Analysis
    def analyse_cardinality(self):
        for field in self.index:
            self.schema[field]["cardinality"] = len(self.index[field])
        return self.schema

    # def disable_high_cardinality(self,threshold):
    #     self.analyse_cardinality()
    #     for field in self.schema:
    #         if self.schema[field]["cardinality"]>threshold:
    #             self.schema[field]["indexed"]=False

## Testing

In [26]:
docs = [
    {
        "name": "Alice",
        "age": 25,
        "salary": 50000,
        "country": "India",
        "active": True,
        "joining": "2023-01-15"
    },
    {
        "name": "Bob",
        "age": 30,
        "salary": 70000,
        "country": "USA",
        "active": False,
        "joining": "2022-06-01"
    },
    {
        "name": "Charlie",
        "age": 35,
        "salary": 60000,
        "country": "India",
        "active": True,
        "joining": "2021-12-20"
    },
    {
        "name": "David",
        "age": 28,
        "salary": 80000,
        "country": "Canada",
        "active": True,
        "joining": "2024-03-10"
    },
]

In [27]:
engine = SearchEngine()

engine.index_documents(docs)

In [28]:
print(engine.tokenize('age between (16,25)'))

['age', 'BETWEEN', '(', '16', ',', '25', ')']


In [29]:
print(engine.tokenize('country NOT IN ("India","USA")'))

['country', 'NOT IN', '(', 'India', ',', 'USA', ')']


In [30]:
print(engine.tokenize('country = "United States"'))

['country', '=', 'United States']


In [31]:
tree = engine.parse('country NOT IN ("India","USA")')

print(tree.field)
print(tree.comparison)
print(tree.value)

country
NOT IN
['India', 'USA']


In [32]:
tree = engine.parse(
    "age > 25 AND country = 'India'"
)

In [33]:
print(tree.operator)

AND


In [34]:
bitmap = engine.evaluate(
    engine.parse("age > 25")
)

print(list(bitmap))

[1, 2, 3]


In [35]:
print(engine.search("age > 25"))

[{'name': 'Bob', 'age': 30, 'salary': 70000, 'country': 'USA', 'active': False, 'joining': '2022-06-01'}, {'name': 'Charlie', 'age': 35, 'salary': 60000, 'country': 'India', 'active': True, 'joining': '2021-12-20'}, {'name': 'David', 'age': 28, 'salary': 80000, 'country': 'Canada', 'active': True, 'joining': '2024-03-10'}]


In [36]:
engine.search(
    'country = "India"'
)

[{'name': 'Alice',
  'age': 25,
  'salary': 50000,
  'country': 'India',
  'active': True,
  'joining': '2023-01-15'},
 {'name': 'Charlie',
  'age': 35,
  'salary': 60000,
  'country': 'India',
  'active': True,
  'joining': '2021-12-20'}]

In [37]:
engine.search(
    'country = "India" AND age > 30'
)

[{'name': 'Charlie',
  'age': 35,
  'salary': 60000,
  'country': 'India',
  'active': True,
  'joining': '2021-12-20'}]

In [38]:
engine.search(
    'country = "India" OR country = "USA"'
)

[{'name': 'Alice',
  'age': 25,
  'salary': 50000,
  'country': 'India',
  'active': True,
  'joining': '2023-01-15'},
 {'name': 'Bob',
  'age': 30,
  'salary': 70000,
  'country': 'USA',
  'active': False,
  'joining': '2022-06-01'},
 {'name': 'Charlie',
  'age': 35,
  'salary': 60000,
  'country': 'India',
  'active': True,
  'joining': '2021-12-20'}]

In [39]:
docs = [
{"name":"Alice","age":21,"salary":45000,"country":"India","department":"HR","active":True,"joining":"2022-01-15"},
{"name":"Bob","age":30,"salary":70000,"country":"USA","department":"Engineering","active":False,"joining":"2022-06-01"},
{"name":"Charlie","age":35,"salary":65000,"country":"India","department":"Engineering","active":True,"joining":"2021-09-20"},
{"name":"David","age":28,"salary":80000,"country":"Canada","department":"Sales","active":True,"joining":"2023-03-11"},
{"name":"Eva","age":25,"salary":52000,"country":"Germany","department":"HR","active":False,"joining":"2020-07-18"},
{"name":"Frank","age":40,"salary":95000,"country":"USA","department":"Management","active":True,"joining":"2019-11-09"},
{"name":"Grace","age":32,"salary":62000,"country":"India","department":"Marketing","active":True,"joining":"2021-05-22"},
{"name":"Henry","age":27,"salary":58000,"country":"UK","department":"Engineering","active":False,"joining":"2022-08-10"},
{"name":"Isabella","age":29,"salary":72000,"country":"Canada","department":"Sales","active":True,"joining":"2023-01-05"},
{"name":"Jack","age":31,"salary":68000,"country":"India","department":"Engineering","active":True,"joining":"2022-12-12"},
{"name":"Karen","age":24,"salary":48000,"country":"Germany","department":"HR","active":False,"joining":"2020-10-30"},
{"name":"Leo","age":45,"salary":110000,"country":"USA","department":"Management","active":True,"joining":"2018-04-17"},
{"name":"Mia","age":26,"salary":56000,"country":"India","department":"Marketing","active":True,"joining":"2021-06-28"},
{"name":"Nathan","age":38,"salary":88000,"country":"Canada","department":"Engineering","active":False,"joining":"2019-09-09"},
{"name":"Olivia","age":22,"salary":47000,"country":"UK","department":"Sales","active":True,"joining":"2023-02-14"},
{"name":"Peter","age":33,"salary":76000,"country":"India","department":"Engineering","active":True,"joining":"2022-05-19"},
{"name":"Queen","age":37,"salary":90000,"country":"Germany","department":"Management","active":True,"joining":"2019-08-08"},
{"name":"Ryan","age":29,"salary":61000,"country":"USA","department":"Marketing","active":False,"joining":"2020-01-11"},
{"name":"Sophia","age":34,"salary":83000,"country":"Canada","department":"Engineering","active":True,"joining":"2021-03-15"},
{"name":"Tom","age":41,"salary":99000,"country":"India","department":"Management","active":False,"joining":"2018-12-01"},
{"name":"Uma","age":23,"salary":43000,"country":"UK","department":"HR","active":True,"joining":"2023-06-01"},
{"name":"Victor","age":36,"salary":87000,"country":"USA","department":"Sales","active":True,"joining":"2020-09-09"},
{"name":"Wendy","age":28,"salary":59000,"country":"India","department":"Engineering","active":True,"joining":"2022-11-11"},
{"name":"Xavier","age":39,"salary":93000,"country":"Germany","department":"Management","active":False,"joining":"2019-02-20"},
{"name":"Yara","age":27,"salary":54000,"country":"Canada","department":"Marketing","active":True,"joining":"2021-10-10"},
{"name":"Zack","age":42,"salary":102000,"country":"USA","department":"Management","active":True,"joining":"2018-01-01"},
{"name":"Aaron","age":20,"salary":41000,"country":"India","department":"HR","active":False,"joining":"2023-07-01"},
{"name":"Bella","age":26,"salary":55000,"country":"UK","department":"Engineering","active":True,"joining":"2022-04-12"},
{"name":"Chris","age":31,"salary":71000,"country":"Canada","department":"Sales","active":False,"joining":"2021-08-18"},
{"name":"Diana","age":35,"salary":85000,"country":"Germany","department":"Management","active":True,"joining":"2020-06-06"},
{"name":"Ethan","age":29,"salary":60000,"country":"India","department":"Marketing","active":True,"joining":"2022-03-03"},
{"name":"Fiona","age":24,"salary":50000,"country":"USA","department":"HR","active":False,"joining":"2021-12-12"},
{"name":"George","age":37,"salary":91000,"country":"Canada","department":"Engineering","active":True,"joining":"2019-07-14"},
{"name":"Hannah","age":33,"salary":77000,"country":"UK","department":"Sales","active":True,"joining":"2020-02-02"},
{"name":"Ian","age":30,"salary":69000,"country":"India","department":"Engineering","active":False,"joining":"2022-09-09"},
{"name":"Julia","age":28,"salary":63000,"country":"Germany","department":"Marketing","active":True,"joining":"2021-04-04"},
{"name":"Kevin","age":43,"salary":108000,"country":"USA","department":"Management","active":True,"joining":"2018-10-10"},
{"name":"Lily","age":25,"salary":53000,"country":"Canada","department":"HR","active":False,"joining":"2023-05-05"},
{"name":"Mark","age":34,"salary":82000,"country":"India","department":"Sales","active":True,"joining":"2020-11-11"},
{"name":"Nina","age":27,"salary":57000,"country":"Germany","department":"Engineering","active":True,"joining":"2022-02-22"},
{"name":"Oscar","age":32,"salary":74000,"country":"USA","department":"Engineering","active":False,"joining":"2021-01-01"},
{"name":"Paula","age":23,"salary":46000,"country":"India","department":"HR","active":True,"joining":"2023-04-04"},
{"name":"Quentin","age":38,"salary":92000,"country":"Canada","department":"Management","active":True,"joining":"2019-05-15"},
{"name":"Rose","age":26,"salary":56000,"country":"UK","department":"Marketing","active":False,"joining":"2022-07-07"},
{"name":"Sam","age":31,"salary":70000,"country":"India","department":"Engineering","active":True,"joining":"2021-11-30"},
{"name":"Tina","age":36,"salary":86000,"country":"Germany","department":"Sales","active":True,"joining":"2020-08-08"},
{"name":"Umar","age":29,"salary":62000,"country":"USA","department":"Marketing","active":False,"joining":"2022-10-20"},
{"name":"Vera","age":40,"salary":97000,"country":"Canada","department":"Management","active":True,"joining":"2018-03-03"},
{"name":"Will","age":28,"salary":61000,"country":"India","department":"Engineering","active":True,"joining":"2023-01-15"},
{"name":"Zoe","age":33,"salary":78000,"country":"UK","department":"Sales","active":False,"joining":"2021-02-28"},
]

In [40]:
newengine=SearchEngine()

In [41]:
newengine.index_documents(docs)

In [42]:
def check(query, expected):
    try:
        actual = sorted(newengine.search(query), key=lambda d: d["name"])
        expected = sorted(expected, key=lambda d: d["name"])

        if actual == expected:
            print(f"✅ PASS: {query}")
        else:
            print(f"❌ FAIL: {query}")
            print(f"Expected {len(expected)} docs, got {len(actual)} docs")
            print("Expected:", [d["name"] for d in expected])
            print("Actual  :", [d["name"] for d in actual])
    except Exception as e:
        print(f"💥 EXCEPTION: {query}")
        print(type(e).__name__, e)

In [43]:
#Numeric Quries

check("age = 30",[d for d in docs if d["age"] == 30])
check("age > 30",[d for d in docs if d["age"] > 30])
check("age >= 30",[d for d in docs if d["age"] >= 30])
check("age < 30",[d for d in docs if d["age"] < 30])
check("age <= 30",[d for d in docs if d["age"] <= 30])
check("salary > 70000",[d for d in docs if d["salary"] > 70000])
check("salary >= 70000",[d for d in docs if d["salary"] >= 70000])
check("salary < 70000",[d for d in docs if d["salary"] < 70000])
check("salary <= 70000",[d for d in docs if d["salary"] <= 70000])
check("salary BETWEEN 60000 AND 80000",[d for d in docs if 60000 <= d["salary"] <= 80000])
check("salary BETWEEN 25 AND 35",[d for d in docs if 25 <= d["salary"] <= 35])

✅ PASS: age = 30
✅ PASS: age > 30
✅ PASS: age >= 30
✅ PASS: age < 30
✅ PASS: age <= 30
✅ PASS: salary > 70000
✅ PASS: salary >= 70000
✅ PASS: salary < 70000
✅ PASS: salary <= 70000
✅ PASS: salary BETWEEN 60000 AND 80000
✅ PASS: salary BETWEEN 25 AND 35


In [44]:
#categorical queries
check('country = "India"',[d for d in docs if d["country"] == "India"])
check(
    'department = "Engineering"',
    [d for d in docs if d["department"] == "Engineering"]
)
check(
    "active = true",
    [d for d in docs if d["active"]]
)
check(
    'country IN ("India","USA")',
    [d for d in docs if d["country"] in ["India", "USA"]]
)
check(
    'department IN ("HR","Sales")',
    [d for d in docs if d["department"] in ["HR", "Sales"]]
)
check(
    'country NOT IN ("India","USA")',
    [d for d in docs if d["country"] not in ["India", "USA"]]
)
check(
    'country = "India" AND department = "Engineering"',
    [
        d for d in docs
        if d["country"] == "India"
        and d["department"] == "Engineering"
    ]
)
check(
    'country = "India" OR country = "Canada"',
    [
        d for d in docs
        if d["country"] == "India"
        or d["country"] == "Canada"
    ]
)
check(
    'NOT country = "India"',
    [
        d for d in docs
        if d["country"] != "India"
    ]
)

check(
    'country = "India" AND active = True',
    [
        d for d in docs
        if d["country"] == "India"
        and d["active"] == True
    ]
)
check(
    '(country = "India" OR country = "USA") AND department = "Engineering"',
    [
        d for d in docs
        if (
            d["country"] == "India"
            or d["country"] == "USA"
        )
        and d["department"] == "Engineering"
    ]
)
check(
    'NOT (country = "India" OR department = "HR") AND active=true',
    [
        d for d in docs
        if not (
            d["country"] == "India"
            or d["department"] == "HR"
        )
        and d["active"] == True
    ]
)

✅ PASS: country = "India"
✅ PASS: department = "Engineering"
✅ PASS: active = true
✅ PASS: country IN ("India","USA")
✅ PASS: department IN ("HR","Sales")
✅ PASS: country NOT IN ("India","USA")
✅ PASS: country = "India" AND department = "Engineering"
✅ PASS: country = "India" OR country = "Canada"
✅ PASS: NOT country = "India"
✅ PASS: country = "India" AND active = True
✅ PASS: (country = "India" OR country = "USA") AND department = "Engineering"
✅ PASS: NOT (country = "India" OR department = "HR") AND active=true


In [45]:
check(
    'joining = "2022-01-15"',
    [d for d in docs if d["joining"] == "2022-01-15"]
)
check(
    'joining > "2022-01-01"',
    [d for d in docs if d["joining"] > "2022-01-01"]
)
check(
    'joining >= "2022-01-01"',
    [d for d in docs if d["joining"] >= "2022-01-01"]
)
check(
    'joining < "2022-01-01"',
    [d for d in docs if d["joining"] < "2022-01-01"]
)
check(
    'joining <= "2022-01-01"',
    [d for d in docs if d["joining"] <= "2022-01-01"]
)
check(
    'joining BETWEEN "2021-01-01" AND "2022-12-31"',
    [
        d for d in docs
        if "2021-01-01" <= d["joining"] <= "2022-12-31"
    ]
)
check(
    'joining IN ("2022-01-15","2023-03-11")',
    [
        d for d in docs
        if d["joining"] in ["2022-01-15", "2023-03-11"]
    ]
)
check(
    'joining NOT IN ("2022-01-15","2023-03-11")',
    [
        d for d in docs
        if d["joining"] not in ["2022-01-15", "2023-03-11"]
    ]
)
check(
    'joining > "2022-01-01" AND country = "India"',
    [
        d for d in docs
        if d["joining"] > "2022-01-01"
        and d["country"] == "India"
    ]
)

✅ PASS: joining = "2022-01-15"
✅ PASS: joining > "2022-01-01"
✅ PASS: joining >= "2022-01-01"
✅ PASS: joining < "2022-01-01"
✅ PASS: joining <= "2022-01-01"
✅ PASS: joining BETWEEN "2021-01-01" AND "2022-12-31"
✅ PASS: joining IN ("2022-01-15","2023-03-11")
✅ PASS: joining NOT IN ("2022-01-15","2023-03-11")
✅ PASS: joining > "2022-01-01" AND country = "India"


In [46]:
#Mixed Queries
check(
    'country = "India" AND age > 30',
    [
        d for d in docs
        if d["country"] == "India" and d["age"] > 30
    ]
)
check(
    'department = "Engineering" AND salary > 70000',
    [
        d for d in docs
        if d["department"] == "Engineering" and d["salary"] > 70000
    ]
)
check(
    'country = "USA" OR salary > 90000',
    [
        d for d in docs
        if d["country"] == "USA" or d["salary"] > 90000
    ]
)
check(
    'country = "India" AND salary BETWEEN 60000 AND 80000',
    [
        d for d in docs
        if d["country"] == "India"
        and 60000 <= d["salary"] <= 80000
    ]
)
check(
    '(country = "India" OR country = "Canada") AND age < 30',
    [
        d for d in docs
        if (d["country"] == "India" or d["country"] == "Canada")
        and d["age"] < 30
    ]
)
check(
    'NOT (country = "Germany") AND salary > 60000',
    [
        d for d in docs
        if not (d["country"] == "Germany")
        and d["salary"] > 60000
    ]
)
check(
    '(department = "Engineering" OR department = "Sales") AND active = true',
    [
        d for d in docs
        if (d["department"] == "Engineering" or d["department"] == "Sales")
        and d["active"]
    ]
)
check(
    '(country = "India" AND department = "Engineering") OR salary > 100000',
    [
        d for d in docs
        if (d["country"] == "India" and d["department"] == "Engineering")
        or d["salary"] > 100000
    ]
)

✅ PASS: country = "India" AND age > 30
✅ PASS: department = "Engineering" AND salary > 70000
✅ PASS: country = "USA" OR salary > 90000
✅ PASS: country = "India" AND salary BETWEEN 60000 AND 80000
✅ PASS: (country = "India" OR country = "Canada") AND age < 30
✅ PASS: NOT (country = "Germany") AND salary > 60000
✅ PASS: (department = "Engineering" OR department = "Sales") AND active = true
✅ PASS: (country = "India" AND department = "Engineering") OR salary > 100000


In [47]:
#stress tests
check(
    'country IN ("India","USA") AND age >= 25',
    [
        d for d in docs
        if d["country"] in ("India", "USA")
        and d["age"] >= 25
    ]
)
check(
    'country NOT IN ("India","USA") OR salary < 50000',
    [
        d for d in docs
        if d["country"] not in ("India", "USA")
        or d["salary"] < 50000
    ]
)
check(
    'NOT (department = "HR" OR active = false)',
    [
        d for d in docs
        if not (d["department"] == "HR" or d["active"] == False)
    ]
)
check(
    '(salary BETWEEN 50000 AND 80000 AND age > 25) OR country = "Canada"',
    [
        d for d in docs
        if (50000 <= d["salary"] <= 80000 and d["age"] > 25)
        or d["country"] == "Canada"
    ]
)

✅ PASS: country IN ("India","USA") AND age >= 25
✅ PASS: country NOT IN ("India","USA") OR salary < 50000
✅ PASS: NOT (department = "HR" OR active = false)
✅ PASS: (salary BETWEEN 50000 AND 80000 AND age > 25) OR country = "Canada"


In [48]:
#edges cases
check(
    'country = "Mars"',
    [d for d in docs if d["country"] == "Mars"]
)

check(
    "salary > 1000000",
    [d for d in docs if d["salary"] > 1000000]
)

check(
    "age < 0",
    [d for d in docs if d["age"] < 0]
)

check(
    'country IN ("India")',
    [d for d in docs if d["country"] in ("India",)]
)

check(
    'country NOT IN ("India")',
    [d for d in docs if d["country"] not in ("India",)]
)

check(
    "salary BETWEEN 70000 AND 70000",
    [d for d in docs if 70000 <= d["salary"] <= 70000]
)

check(
    "age >= 18 AND age <= 60",
    [d for d in docs if d["age"] >= 18 and d["age"] <= 60]
)

check(
    "NOT (salary > 90000)",
    [d for d in docs if not (d["salary"] > 90000)]
)

✅ PASS: country = "Mars"
✅ PASS: salary > 1000000
✅ PASS: age < 0
✅ PASS: country IN ("India")
✅ PASS: country NOT IN ("India")
✅ PASS: salary BETWEEN 70000 AND 70000
✅ PASS: age >= 18 AND age <= 60
✅ PASS: NOT (salary > 90000)


In [49]:
#parser stress tests
check(
    '(((country = "India")))',
    [
        d for d in docs
        if d["country"] == "India"
    ]
)
check(
    '(country = "India" AND (department = "Engineering" OR department = "HR"))',
    [
        d for d in docs
        if d["country"] == "India"
        and (
            d["department"] == "Engineering"
            or d["department"] == "HR"
        )
    ]
)
check(
    'NOT (country = "India" AND active = true)',
    [
        d for d in docs
        if not (
            d["country"] == "India"
            and d["active"] == True
        )
    ]
)
check(
    '(age > 25 AND salary < 90000) OR (country = "USA")',
    [
        d for d in docs
        if (
            d["age"] > 25
            and d["salary"] < 90000
        )
        or d["country"] == "USA"
    ]
)
check(
    'NOT ((country = "India") OR (department = "HR"))',
    [
        d for d in docs
        if not (
            d["country"] == "India"
            or d["department"] == "HR"
        )
    ]
)
check(
    'NOT country = "India" OR age > 30',
    [
        d for d in docs
        if (not (d["country"] == "India"))
        or d["age"] > 30
    ]
)
check(
    'NOT (NOT (country = "India"))',
    [
        d for d in docs
        if not (not (d["country"] == "India"))
    ]
)
check(
    '(((country = "India") AND ((department = "Engineering"))))',
    [
        d for d in docs
        if d["country"] == "India"
        and d["department"] == "Engineering"
    ]
)
check(
    '((country = "India" OR country = "USA") AND (salary > 60000 AND age < 40))',
    [
        d for d in docs
        if (
            d["country"] == "India"
            or d["country"] == "USA"
        )
        and (
            d["salary"] > 60000
            and d["age"] < 40
        )
    ]
)

✅ PASS: (((country = "India")))
✅ PASS: (country = "India" AND (department = "Engineering" OR department = "HR"))
✅ PASS: NOT (country = "India" AND active = true)
✅ PASS: (age > 25 AND salary < 90000) OR (country = "USA")
✅ PASS: NOT ((country = "India") OR (department = "HR"))
✅ PASS: NOT country = "India" OR age > 30
✅ PASS: NOT (NOT (country = "India"))
✅ PASS: (((country = "India") AND ((department = "Engineering"))))
✅ PASS: ((country = "India" OR country = "USA") AND (salary > 60000 AND age < 40))


## Testing for randomized data

In [50]:
import random
from datetime import datetime, timedelta

countries = ["India", "USA", "Canada", "Germany", "Japan"]
departments = ["Engineering", "HR", "Sales", "Finance", "Marketing"]

def random_date():
    start = datetime(2020, 1, 1)
    end = datetime(2024, 12, 31)
    delta = (end - start).days
    return (start + timedelta(days=random.randint(0, delta))).strftime("%Y-%m-%d")

def generate_dataset(n=50):
    docs = []

    for i in range(n):
        docs.append({
            "name": f"Employee{i}",
            "age": random.randint(18, 60),
            "salary": random.randint(30000, 120000),
            "country": random.choice(countries),
            "department": random.choice(departments),
            "active": random.choice([True, False]),
            "joining": random_date()
        })

    return docs

In [51]:
def newcheck(query, expected,bestengine):
    try:
        actual = sorted(bestengine.search(query), key=lambda d: d["name"])
        expected = sorted(expected, key=lambda d: d["name"])

        if actual == expected:
            print(f"✅ PASS: {query}")
        else:
            print(f"❌ FAIL: {query}")
            print(f"Expected {len(expected)} docs, got {len(actual)} docs")
            print("Expected:", [d["name"] for d in expected])
            print("Actual  :", [d["name"] for d in actual])
    except Exception as e:
        print(f"💥 EXCEPTION: {query}")
        print(type(e).__name__, e)

In [52]:
def test_dataset(docs):

    bestengine = SearchEngine()

    bestengine.index_documents(docs)

    newcheck('country = "India"',
          [d for d in docs if d["country"] == "India"],bestengine)

    newcheck("age > 30",
          [d for d in docs if d["age"] > 30],bestengine)

    newcheck("salary BETWEEN 50000 AND 80000",
          [d for d in docs if 50000 <= d["salary"] <= 80000],bestengine)

    newcheck('country IN ("India","USA")',
          [d for d in docs if d["country"] in ["India","USA"]],bestengine)

    newcheck('country NOT IN ("India","USA")',
          [d for d in docs if d["country"] not in ["India","USA"]],bestengine)

    newcheck(
        '(country = "India" OR country = "USA") AND age > 30',
        [
            d for d in docs
            if (d["country"] == "India" or d["country"] == "USA")
            and d["age"] > 30
        ],bestengine
    )

    newcheck(
        'NOT (country = "Germany")',
        [
            d for d in docs
            if d["country"] != "Germany"
        ],bestengine
    )

In [53]:

NUM_DATASETS = 5000

for i in range(NUM_DATASETS):
    random.seed(i)      # Dataset i is always the same

    print(f"Testing dataset {i+1}/{NUM_DATASETS}")

    docs = generate_dataset(50)

    try:
        test_dataset(docs)
    except Exception as e:
        print(f"\n❌ FAILED on dataset {i+1}")
        print(f"Seed: {i}")
        print(e)
        break
else:
    print(f"\n🎉 All {NUM_DATASETS} datasets passed!")

Testing dataset 1/5000
✅ PASS: country = "India"
✅ PASS: age > 30
✅ PASS: salary BETWEEN 50000 AND 80000
✅ PASS: country IN ("India","USA")
✅ PASS: country NOT IN ("India","USA")
✅ PASS: (country = "India" OR country = "USA") AND age > 30
✅ PASS: NOT (country = "Germany")
Testing dataset 2/5000
✅ PASS: country = "India"
✅ PASS: age > 30
✅ PASS: salary BETWEEN 50000 AND 80000
✅ PASS: country IN ("India","USA")
✅ PASS: country NOT IN ("India","USA")
✅ PASS: (country = "India" OR country = "USA") AND age > 30
✅ PASS: NOT (country = "Germany")
Testing dataset 3/5000
✅ PASS: country = "India"
✅ PASS: age > 30
✅ PASS: salary BETWEEN 50000 AND 80000
✅ PASS: country IN ("India","USA")
✅ PASS: country NOT IN ("India","USA")
✅ PASS: (country = "India" OR country = "USA") AND age > 30
✅ PASS: NOT (country = "Germany")
Testing dataset 4/5000
✅ PASS: country = "India"
✅ PASS: age > 30
✅ PASS: salary BETWEEN 50000 AND 80000
✅ PASS: country IN ("India","USA")
✅ PASS: country NOT IN ("India","USA")
✅ 

In [54]:
import time

docs = generate_dataset(100000)

greatengine = SearchEngine()

start = time.perf_counter()
greatengine.index_documents(docs)
end = time.perf_counter()

elapsed = end - start

print(f"Indexed {len(docs)} docs in {elapsed:.3f}s")
print(f"Throughput: {len(docs)/elapsed:,.0f} docs/sec")

Indexed 100000 docs in 4.701s
Throughput: 21,272 docs/sec


In [55]:
import time

query = '(country = "India" OR country = "USA") AND salary > 70000'

# Warm-up
engine.search(query)

start = time.perf_counter()

for _ in range(1000):
    engine.search(query)

end = time.perf_counter()

avg_ms = (end - start) * 1000 / 1000

print(f"Average query time: {avg_ms:.3f} ms")

Average query time: 0.203 ms
